In [34]:
import pandas as pd
import numpy as np
import glob


order_files = [
    "data/orders_2024_01.csv",
    "data/orders_2024_02.csv",
    "data/orders_2024_03.csv",
]

df_orders_01 = pd.read_csv(order_files[0], parse_dates=["order_datetime"])
df_orders_02 = pd.read_csv(order_files[1], parse_dates=["order_datetime"])
df_orders_03 = pd.read_csv(order_files[2], parse_dates=["order_datetime"])

print("orders_2024_01:", df_orders_01.shape)
print("orders_2024_02:", df_orders_02.shape)
print("orders_2024_03:", df_orders_03.shape)

df_orders = pd.concat(
    [df_orders_01, df_orders_02, df_orders_03],
    ignore_index=True,
)

print("\nОбъединённые заказы:")
print(df_orders.info())
print(df_orders.head())

sheets = pd.read_excel(
    "data/products.xlsx",
    sheet_name=["products", "categories"],
)

df_products = sheets["products"]
df_categories = sheets["categories"]

print("\nТовары:")
print(df_products.head())
print(df_products.info())

print("\nКатегории:")
print(df_categories.head())
print(df_categories.info())

orders_2024_01: (5000, 7)
orders_2024_02: (5000, 7)
orders_2024_03: (5000, 7)

Объединённые заказы:
<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order_id        15000 non-null  int64         
 1   user_id         15000 non-null  int64         
 2   order_datetime  15000 non-null  datetime64[us]
 3   product_id      15000 non-null  int64         
 4   price           15000 non-null  float64       
 5   quantity        15000 non-null  int64         
 6   status          15000 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(4), str(1)
memory usage: 761.8 KB
None
   order_id  user_id      order_datetime  product_id     price  quantity  \
0         1     1044 2024-01-03 18:39:26          26  14038.15         1   
1         2     1079 2024-01-03 17:45:43          16   9172.60         1   
2         3     1041 20

In [35]:
print("Задание 1: Чтение данных")
print(f"Всего заказов: {df_orders.shape[0]} строк, {df_orders.shape[1]} столбцов")
print(f"Товаров: {df_products.shape[0]} строк")
print(f"Категорий: {df_categories.shape[0]} строк")

Задание 1: Чтение данных
Всего заказов: 15000 строк, 7 столбцов
Товаров: 30 строк
Категорий: 5 строк


In [36]:
print("Задание 2: Объединение и приведение типов")

df_orders_prod["price"] = df_orders_prod["price"].astype("float32")
df_orders_prod["user_id"] = df_orders_prod["user_id"].astype("int32")
df_orders_prod["product_id"] = df_orders_prod["product_id"].astype("int32")
df_orders_prod["category_id"] = df_orders_prod["category_id"].astype("Int32") 
print(df_orders_prod.dtypes)
print(df_orders_prod.head())

Задание 2: Объединение и приведение типов
order_id                    int64
user_id                     int32
order_datetime     datetime64[us]
product_id                  int32
price                     float32
quantity                    int64
status                        str
product_name                  str
category_id                 Int32
category_name_x               str
weight_kg                 float64
category_name_y               str
description                   str
revenue                   float64
dtype: object
   order_id  user_id      order_datetime  product_id         price  quantity  \
0         1     1044 2024-01-03 18:39:26          26  14038.150391         1   
1         2     1079 2024-01-03 17:45:43          16   9172.599609         1   
2         3     1041 2024-01-06 22:46:38          25   9035.259766         2   
3         4     1083 2024-01-08 21:03:51           9  15234.330078         3   
4         5     1068 2024-01-03 23:26:53          24   4134.379883  

In [37]:
print("Задание 3: Выручка по (user_id, category_id)")

df_orders_prod["revenue"] = df_orders_prod["price"] * df_orders_prod["quantity"]

revenue_by_user_cat = (
    df_orders_prod
    .groupby(["user_id", "category_id"], as_index=False)
    .agg(revenue_sum=("revenue", "sum"))
)

print(revenue_by_user_cat.head(10))
print(f"Всего комбинаций: {revenue_by_user_cat.shape[0]}")

Задание 3: Выручка по (user_id, category_id)
   user_id  category_id   revenue_sum
0     1001            1  1.079367e+06
1     1001            2  5.109530e+05
2     1001            3  1.212973e+06
3     1001            4  3.941912e+05
4     1001            5  1.238283e+06
5     1002            1  4.306263e+05
6     1002            2  4.098305e+05
7     1002            3  7.036556e+05
8     1002            4  3.834103e+05
9     1002            5  9.024817e+05
Всего комбинаций: 500


In [38]:
print("Задание 4: Топ-5 категорий на пользователя")

revenue_sorted = revenue_by_user_cat.sort_values(
    ["user_id", "revenue_sum"],
    ascending=[True, False],
)

top5_categories_per_user = (
    revenue_sorted
    .groupby("user_id", as_index=False)
    .head(5)
    .reset_index(drop=True)
)

print(top5_categories_per_user.head(20))
print(f"Строк в результате: {top5_categories_per_user.shape[0]}")

Задание 4: Топ-5 категорий на пользователя
    user_id  category_id   revenue_sum
0      1001            5  1.238283e+06
1      1001            3  1.212973e+06
2      1001            1  1.079367e+06
3      1001            2  5.109530e+05
4      1001            4  3.941912e+05
5      1002            5  9.024817e+05
6      1002            3  7.036556e+05
7      1002            1  4.306263e+05
8      1002            2  4.098305e+05
9      1002            4  3.834103e+05
10     1003            5  1.448524e+06
11     1003            3  8.314840e+05
12     1003            1  7.973570e+05
13     1003            4  5.261685e+05
14     1003            2  4.217329e+05
15     1004            3  1.111600e+06
16     1004            1  9.891132e+05
17     1004            5  8.847382e+05
18     1004            4  5.060325e+05
19     1004            2  2.965466e+05
Строк в результате: 500


In [39]:
print("Задание 5: Сохранение в Parquet")

import pyarrow as pa
import pyarrow.parquet as pq

df_to_save = top5_categories_per_user.copy()
df_to_save["category_id"] = df_to_save["category_id"].astype("int64")
df_to_save["user_id"] = df_to_save["user_id"].astype("int64")

table = pa.Table.from_pandas(df_to_save, preserve_index=False)
pq.write_table(table, "data/top5_categories_per_user.parquet")

check = pq.read_table("data/top5_categories_per_user.parquet").to_pandas()
print(check.head())
print(f"Размер: {check.shape}")

Задание 5: Сохранение в Parquet
   user_id  category_id   revenue_sum
0     1001            5  1.238283e+06
1     1001            3  1.212973e+06
2     1001            1  1.079367e+06
3     1001            2  5.109530e+05
4     1001            4  3.941912e+05
Размер: (500, 3)
